In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import random
import numpy as np
from src.river_sddm import RiverSDDM

def test_sddm_local_drift(incremental = False):
    # 1. Konfiguracja detektora dla klastra
    features = ['sensor_a', 'sensor_b', 'sensor_c']
    detector = RiverSDDM(
        n_bins=10, 
        ref_window_size=200,  # Rozmiar okna referencyjnego [cite: 415]
        cur_window_size=100,  # Rozmiar bieżącego okna (batch p) [cite: 292]
        threshold=0.6,        # Próg dla nagłego dryfu (abrupt drift) [cite: 578]
        test_interval=5,      # Częstotliwość testowania [cite: 243]
        alpha=1.0,
        incremental= incremental                          
    )

    print(f"Rozpoczynam test dla cech: {features}\n")

    # 2. Generowanie strumienia danych
    for t in range(1, 5000):
        # Stan normalny: wszystkie cechy w zakresie [0.0, 0.4]
        if t < 800:
            x = {f: random.uniform(0.0, 0.4) for f in features}
        
        # Symulacja dryfu: tylko 'sensor_b' zmienia zakres na [0.6, 1.0]
        elif t >=800 and t < 2000: 
            x = {
                'sensor_a': random.uniform(0.0, 0.4),
                'sensor_b': random.uniform(0.6, 0.8), # Tu następuje dryf [cite: 104]
                'sensor_c': random.uniform(0.0, 0.4)
            }
        else:
            x = {
                'sensor_a': random.uniform(0.0, 0.4),
                'sensor_b': random.uniform(0.8, 1), # Tu następuje dryf [cite: 104]
                'sensor_c': random.uniform(0.0, 0.4)
            }
        # Aktualizacja detektora (obsługuje x jako dict i opcjonalne y) [cite: 18, 80]
        detector.update(x, y=None) # Test bez etykiet (P(X))

        # 3. Sprawdzanie detekcji
        if detector.drift_detected:
            report = detector.get_drift_report()
            print(f"ALARM: Wykryto dryf lokalny!")
            print(f"  - Krok czasowy: {t}")
            print(f"  - Siła dryfu (Magnitude): {report['magnitude']:.4f}")
            print(f"  - Źródło dryfu (Source): {report['source']}")
            print(f"  - Status okna: Zresetowano (Resetting approach) [cite: 234]")
            print("-" * 40)


In [46]:
test_sddm_local_drift(False)

Rozpoczynam test dla cech: ['sensor_a', 'sensor_b', 'sensor_c']

ALARM: Wykryto dryf lokalny!
  - Krok czasowy: 835
  - Siła dryfu (Magnitude): 0.6106
  - Źródło dryfu (Source): sensor_b
  - Status okna: Zresetowano (Resetting approach) [cite: 234]
----------------------------------------
ALARM: Wykryto dryf lokalny!
  - Krok czasowy: 935
  - Siła dryfu (Magnitude): 1.0360
  - Źródło dryfu (Source): sensor_b
  - Status okna: Zresetowano (Resetting approach) [cite: 234]
----------------------------------------
ALARM: Wykryto dryf lokalny!
  - Krok czasowy: 1040
  - Siła dryfu (Magnitude): 0.6101
  - Źródło dryfu (Source): sensor_a
  - Status okna: Zresetowano (Resetting approach) [cite: 234]
----------------------------------------
ALARM: Wykryto dryf lokalny!
  - Krok czasowy: 1145
  - Siła dryfu (Magnitude): 0.7031
  - Źródło dryfu (Source): sensor_b
  - Status okna: Zresetowano (Resetting approach) [cite: 234]
----------------------------------------
ALARM: Wykryto dryf lokalny!
  - 

In [ ]:
f